# DiariZen + DiCoW — thử checkpoint tác giả trên Colab

Notebook inference trên branch `feat/dicow-author-pipeline`, dùng đúng adapter của project và các revision trong `DICOW.md`. Không train tự động. Chọn runtime GPU trong Colab, upload `colab-bundle.zip` được tạo bằng `python scripts/package_dicow_colab.py --include-samples`.

Checkpoint: `BUT-FIT/DiCoW_v3_MLC`; diarizer: `BUT-FIT/diarizen-wavlm-large-s80-md`. FDDT và STNO nằm trong code tác giả; các target decode tuần tự để giảm VRAM. Đây là thử pretrained, chưa tái huấn luyện paper v1.


In [ ]:
import os, sys, subprocess, pathlib, zipfile, json
from google.colab import files
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload đúng một colab-bundle.zip của project.')
workspace = pathlib.Path('/content')
with zipfile.ZipFile(archives[0]) as bundle:
    for name in bundle.namelist():
        target = (workspace / name).resolve()
        if not target.is_relative_to(workspace) or not name.startswith('meeting_asr/'):
            raise ValueError('Bundle không đúng cấu trúc project.')
    bundle.extractall(workspace)
project = workspace / 'meeting_asr'
os.chdir(project)
os.environ['MEETING_LLM_ENABLED'] = 'false'
subprocess.run(['nvidia-smi'], check=True)


## Cài runtime

Baseline package và runtime tác giả dùng hai môi trường riêng. Runtime tác giả cần tải Torch/CUDA và khoảng 4 GB trọng số DiCoW, nên lần đầu có thể lâu. Không đồng bộ lại `.venv` baseline hoặc thay fork Pyannote vào môi trường baseline.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
subprocess.run([sys.executable, 'scripts/setup_dicow.py', '--python', sys.executable], check=True)


## Hugging Face token

Thêm secret tên `HF_TOKEN` trong Colab và cấp quyền cho notebook. Token cần quyền đọc các model Pyannote/WeSpeaker đã chấp nhận điều kiện. Không ghi token vào code, notebook output hoặc file cấu hình.


In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('HF token đã nạp:', bool(os.environ.get('HF_TOKEN')))


## Chạy một đoạn trước

Bundle có ba đoạn 5 phút: `haveasip_13_giang_oi/sample_5min.wav`, `cosv_duc_ly/sample_first_5min.wav`, `cosv_dung_an/sample_first_5min.wav`. Đổi `AUDIO_FILE` để thử đoạn khác, dùng output riêng. LLM tắt. `--no-cache` đo inference thật; trọng số tải vẫn được dùng lại.


In [ ]:
AUDIO_FILE = 'data/real/haveasip_13_giang_oi/sample_5min.wav'
OUTPUT_DIR = 'results/colab-dicow/haveasip_13'
subprocess.run([
    sys.executable, '-m', 'meeting_asr.cli', 'run', AUDIO_FILE,
    '--config', 'configs/dicow-author.yaml', '--out', OUTPUT_DIR,
    '--device', 'cuda', '--no-cache',
], check=True)


## Xem transcript và số đo thật

Số người/turn là dự đoán, không phải độ chính xác. Peak CUDA allocated khác tổng VRAM. Transcript đầy đủ COSV và phụ đề YouTube tự động không phải nhãn chuẩn của đoạn thử 5 phút; chưa tính WER/cpWER/DER khi chưa nghe xác nhận nhãn tương ứng.


In [ ]:
from IPython.display import display, Audio
import pandas as pd
output = pathlib.Path(OUTPUT_DIR)
minutes = json.loads((output / 'minutes.json').read_text())
run = json.loads((output / 'run.json').read_text())
provenance = json.loads((output / 'dicow-provenance.json').read_text())
display(Audio(filename=AUDIO_FILE))
display(pd.DataFrame(minutes['turns'])[['start', 'end', 'speaker', 'text']])
print(json.dumps({
    'predicted_speakers': minutes['num_speakers'],
    'predicted_turns': len(minutes['turns']),
    'elapsed_sec': run['result']['elapsed_sec'],
    'rtf': run['result']['rtf'],
    'diarization_peak_cuda_allocated_gib': provenance.get('diarization_peak_cuda_allocated_bytes', 0) / 2**30,
    'asr_peak_cuda_allocated_gib': provenance.get('asr_peak_cuda_allocated_bytes', 0) / 2**30,
    'packages': provenance['packages'],
}, ensure_ascii=False, indent=2))


In [ ]:
import shutil
archive = shutil.make_archive(str(output.resolve()) + '-artifacts', 'zip', root_dir=output)
files.download(archive)


## Trước khi fine-tune

Chỉ cân nhắc fine-tune sau khi xác định lỗi chữ hay lỗi diarization. Chuẩn bị audio, transcript, speaker activity theo thời gian và split train/dev/test độc lập; nhãn tự động phải được nghe xác nhận. Dùng training repo của tác giả `BUTSpeechFIT/TS-ASR-Whisper` và checkpoint MLC làm điểm bắt đầu. Notebook này chưa chạy fine-tuning hoặc xác nhận ngân sách compute units đủ cho training.
